# 05 · Fine-tune Qwen3.5-9B: SFT LoRA BF16

Train một LoRA adapter cho Qwen3.5-9B bằng **supervised fine-tuning**. Dữ liệu mặc định: bộ SFT trong `data/` (`data/data/train.jsonl`, `dev.jsonl`,
`sft_manifest.json`: 6.363 / 1.064 dòng, đã chia train / dev theo shop), hoặc lời giải đúng của tập train từ `04_training_data`.
Cùng công thức chạy một lệnh, không cần notebook: `bash training/train_a6000.sh`.
Script: `training/train_lora_bf16.py`; kiểm thử trên một Qwen3.5 tí hon: `tests/test_train_lora_bf16.py`.

| Thành phần | Thiết lập | Lý do |
|---|---|---|
| Mô hình gốc | Qwen3.5-9B nạp **bf16** đầy đủ, đóng băng | adapter khớp đúng trọng số vLLM dùng lúc suy luận. QLoRA 4-bit (v1-v3, `training/train_lora.py`) học cả phần bù sai số lượng tử NF4 vốn không có lúc suy luận |
| LoRA | r 16, alpha 32, dropout 0 trên 248 module: q/k/v/o (8 lớp full attention), in_proj_qkv/z/b/a + out_proj (24 lớp Gated DeltaNet), gate/up/down (MLP cả 32 lớp) | cùng module và tên tensor với `adapters/sql_rft_v3`: vLLM (`--lora`, max rank 64) dùng ngay, tiếp tục được từ v3 |
| Prompt | chat template của chính model, `enable_thinking` theo từng dòng; prompt và trả lời tokenize riêng | đúng chuỗi token vLLM nhận lúc suy luận |
| Loss | cross-entropy trung bình trên phần trả lời (+ `<\|im_end\|>`) của mỗi dòng, prompt mask -100 | prompt (rulebook + schema) chiếm ~96% token |
| LM head | chỉ trên các vị trí dự đoán token trả lời, tính theo khối (`--ce-chunk`) | cùng loss và gradient, không tạo tensor logit [độ dài trả lời × 248k] float32 |
| Tối ưu | AdamW fused, lr 1e-4 (5e-5 khi tiếp tục adapter), cosine, warmup 3%, clip 1.0, 16 dòng mỗi bước | |
| Chọn checkpoint | dev loss trên các shop không train, adapter cuối = checkpoint dev loss thấp nhất, dừng sớm | dev loss trên một tập con cố định `DEV_MAX_ROWS` dòng: mỗi dòng ~13k token, đánh giá cả dev tốn ~30% thời gian train |
| Bộ nhớ | gradient checkpointing, micro-batch 1, dòng dài hơn `MAX_LEN` bị loại (không cắt) | dòng bị cắt mất phần trả lời ở cuối |

**Phần cứng:** GPU bf16 (Ampere trở lên) **từ 40 GB** mỗi tiến trình: A100 40/80 GB, L40S / A6000 / RTX 6000 Ada 48 GB, H100. Trọng số bf16 chiếm
~18 GiB, phần còn lại cho activation của các dòng 9k-32k token. GPU 24 GB không đủ cho BF16 với dữ liệu này. Nhiều GPU: data parallel
(`torchrun`), mỗi GPU một bản bf16, số dòng mỗi bước = `GRAD_ACCUM` × số GPU.

**Các bước:** (1) kiểm tra dữ liệu bằng tokenizer, chọn `MAX_LEN` → (2) kiểm tra rò rỉ train / dev / holdout → (3) kiểm tra kernel fla →
(4) preflight bộ nhớ trên các dòng dài nhất → (5) train, checkpoint định kỳ, chạy lại để tiếp tục → (6) đường loss, kiểm tra adapter → (7) dùng adapter.

In [ ]:
# Setup: tìm checkout của repo này, nạp datamate.notebook, phát hiện dữ liệu, GPU, thư mục model và runs.
#   local   mở notebook trong notebooks/ của checkout (hoặc đặt biến môi trường DATAMATE_REPO), không cần cài thêm.
#   Kaggle  đính kèm bản sao checkout làm dataset (Add data) cùng dữ liệu BTC (dataset riêng tư), bật Internet để cài gói.
#   Colab   chép checkout vào MyDrive/datamate/code, dữ liệu BTC vào MyDrive/datamate/data.
# Kaggle / Colab: code được chép sang ổ ghi được (input của Kaggle chỉ đọc, Drive chậm); kết quả ghi vào env.runs, được giữ sau phiên.
import os, shutil, sys
from pathlib import Path

MARKER = "datamate/notebook.py"


def find_checkout() -> Path:
    if os.environ.get("DATAMATE_REPO"):
        return Path(os.environ["DATAMATE_REPO"]).resolve()
    for c in (Path.cwd(), Path.cwd().parent):
        if (c / MARKER).exists():
            return c.resolve()
    if Path("/kaggle/input").exists():
        found, work = sorted(p.parents[1] for p in Path("/kaggle/input").glob(f"**/{MARKER}")), Path("/tmp/datamate")
    elif "COLAB_RELEASE_TAG" in os.environ:
        from google.colab import drive
        drive.mount("/content/drive")
        found, work = [p for p in [Path("/content/drive/MyDrive/datamate/code")] if (p / MARKER).exists()], Path("/content/datamate")
    else:
        found, work = [], None
    if not found:
        raise FileNotFoundError("không tìm thấy checkout datamate: mở notebook trong notebooks/ của repo, hoặc đặt DATAMATE_REPO")
    if not (work / MARKER).exists():
        shutil.copytree(found[0], work, ignore=shutil.ignore_patterns(".git", "runs", "models", ".venv*", "__pycache__"))
    return work


root = find_checkout()
os.chdir(root)
sys.path.insert(0, str(root))
from datamate.notebook import install, setup
install("train")
env = setup()

In [ ]:
import json
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

read = lambda p: [json.loads(l) for l in open(p, encoding="utf-8-sig") if l.strip()]
pd.set_option("display.max_colwidth", 80)

In [ ]:
TRAIN = "data/data/train.jsonl"             # bộ SFT trong data/ (đã chia theo shop, có sft_manifest.json); hoặc runs/sft/<x>/train.jsonl của 04_training_data
DEV = "data/data/dev.jsonl"                 # shop không train: dev loss, chọn checkpoint, dừng sớm. None = fit cuối trên mọi shop
OUT = "runs/lora/qwen35_sft_a6000"          # checkpoint-*/, adapter/, data_report.json, run_manifest.json, nhật ký
INIT_ADAPTER = None                         # None = adapter mới. "sql_rft_v3" = tiếp tục bản đã nộp (v3 đã học câu holdout: holdout không còn sạch)
MAX_LEN = None                              # token mỗi dòng; None = mức nhỏ nhất (16384 .. 40960) giữ được >= KEEP_SHARE số dòng train (data/: 16384, giữ 100%)
KEEP_SHARE = 0.99
R, ALPHA, DROPOUT = 16, 32, 0.0             # r 16 / alpha 32: tiếp tục được từ sql_rft_v3; vLLM nhận tới rank 64
LR = None                                   # None = 1e-4 (adapter mới) hoặc 5e-5 (tiếp tục INIT_ADAPTER)
EPOCHS = 2                                  # checkpoint dev loss tốt nhất được giữ, dừng sớm khi dev loss thôi giảm
MICRO_BATCH, GRAD_ACCUM = 1, 16             # 16 dòng mỗi bước tối ưu trên mỗi GPU (data/: ~398 bước mỗi epoch)
WARMUP_RATIO, SCHEDULER, WEIGHT_DECAY, MAX_GRAD_NORM = 0.03, "cosine", 0.0, 1.0
EVAL_STEPS, SAVE_STEPS = 50, 50             # SAVE_STEPS phải là bội của EVAL_STEPS
EARLY_STOP = 3                              # dừng sau 3 lần đánh giá liên tiếp không giảm dev loss (0 = tắt)
DEV_MAX_ROWS = 320                          # dev loss trên 320 dòng dev cố định (0 = cả dev; data/ có 1.064 dòng ~13k token)
CE_CHUNK = 1024                             # LM head theo khối 1024 vị trí trả lời (0 = head thường, tốn bộ nhớ hơn với câu trả lời dài)
ATTN = "sdpa"                               # 8 lớp full attention: sdpa, hoặc flash_attention_2 khi đã cài flash-attn
NUM_GPUS = None                             # None = mọi GPU thấy được
LIMIT_ROWS, MAX_STEPS = None, -1            # chạy thử nhanh: ví dụ LIMIT_ROWS = 64, MAX_STEPS = 10
PREFLIGHT = True                            # 2 bước trên các dòng dài nhất trước khi train thật: đo đỉnh bộ nhớ ở MAX_LEN
RESUME = True                               # tiếp tục từ checkpoint mới nhất trong OUT (cùng dữ liệu và thiết lập)
SEED = 3407
REPORT_TO = "tensorboard"                   # none | tensorboard | wandb

In [ ]:
train_file, dev_file, out = env.path(TRAIN), env.path(DEV) if DEV else None, env.path(OUT)
for f in [train_file] + ([dev_file] if dev_file else []):
    assert f.exists(), f"{f} chưa có: đặt bộ SFT vào data/data/ hoặc chạy 04_training_data trước"
init = env.adapter(INIT_ADAPTER) if INIT_ADAPTER else None
lr = LR or (5e-5 if init else 1e-4)
venv_train = env.root / ".venv-train/bin/python"
py = str(venv_train) if venv_train.exists() else env.python      # stack train (torch, transformers, peft, fla) tách khỏi vLLM
n_gpu = NUM_GPUS or max(1, len(env.gpus))
train_env = {"PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True", "TOKENIZERS_PARALLELISM": "false"}
print(f"python train: {py}\nGPU: {[f'{g.name} {g.mem_gb} GB' for g in env.gpus] or 'không có'} | BF16 LoRA chạy được ở đây: {env.can_train_bf16} "
      f"(cần bf16 và >= 40 GB mỗi GPU)\nlr {lr} | init {init or 'adapter mới'} | {n_gpu} GPU x {MICRO_BATCH} x {GRAD_ACCUM} = "
      f"{n_gpu * MICRO_BATCH * GRAD_ACCUM} dòng mỗi bước")

### Kiểm tra rò rỉ

- Mọi dòng phải là câu **train** (có đáp án), không bao giờ là câu public / private. Bộ SFT trong `data/` là dữ liệu tổng hợp
  (`metadata.dataset_role = synthetic_train`, shop `syn_shop_*`): không chứa câu nào của BTC, mã câu hỏi và shop nằm ngay trong dòng.
- Train và dev **không chung câu hỏi, không chung shop** (dev loss đo khả năng sang shop mới).
- Không có câu của **holdout** (`splits/holdout600.txt`), để chạy pipeline trên holdout với adapter mới là phép đo sạch. Tiếp tục từ
  `sql_rft_v3` cũng làm holdout không còn sạch (v3 đã học 390 câu holdout).

In [ ]:
holdout_ids = set((env.root / "splits/holdout600.txt").read_text().split())
rows = {"train": read(train_file), "dev": read(dev_file) if dev_file else []}
if any("prompt_messages" in r for rs in rows.values() for r in rs):
    # bộ SFT của data/ (dữ liệu tổng hợp): mã câu hỏi và shop ở ngay trong dòng
    roles = Counter((r.get("metadata") or {}).get("dataset_role") for rs in rows.values() for r in rs)
    assert set(roles) == {"synthetic_train"}, f"có dòng không thuộc dữ liệu tổng hợp để train: {dict(roles)}"
    ids = {s: {r["metadata"]["question_id"] for r in rs} for s, rs in rows.items()}
    shops = {s: {r["business_id"] for r in rs} for s, rs in rows.items()}
else:
    # dữ liệu của 04_training_data: mọi dòng phải là câu train của BTC
    q_shop = {q["id"]: q["business_id"] for q in read(env.split("train") / "questions.jsonl")}
    ids = {s: {str(r["id"]) for r in rs} for s, rs in rows.items()}
    shops = {s: {q_shop.get(i) for i in v} for s, v in ids.items()}
    unknown = (ids["train"] | ids["dev"]) - set(q_shop)
    assert not unknown, f"{len(unknown)} dòng không phải câu train, ví dụ {sorted(unknown)[:3]}"
assert not ids["train"] & ids["dev"], f"train và dev chung câu hỏi: {sorted(ids['train'] & ids['dev'])[:3]}"
assert not shops["train"] & shops["dev"], f"train và dev chung shop: {sorted(shops['train'] & shops['dev'])}"
leak = (ids["train"] | ids["dev"]) & holdout_ids
holdout_clean = not leak and init is None
display(pd.DataFrame({s: {"rows": len(rows[s]), "questions": len(ids[s]), "shops": len(shops[s]),
                          "thinking rows": sum(bool(r.get("thinking", r.get("thinking_mode"))) for r in rows[s]),
                          "holdout questions": len(ids[s] & holdout_ids)} for s in rows}).T)
print("holdout sạch cho adapter này:" , holdout_clean, "" if holdout_clean else
      f"({len(leak)} câu holdout trong dữ liệu)" if leak else "(tiếp tục từ adapter đã học câu holdout)")
del rows   # ~350 MB JSON: trainer tự đọc lại


### 1. Kiểm tra dữ liệu bằng tokenizer

Chỉ cần tokenizer và chat template (không cần trọng số, không cần GPU): mỗi dòng được render đúng như lúc suy luận, kiểm tra định dạng
(khối ```` ```sql ````, đúng một `</think>` ở dòng thinking, không có token đặc biệt trong câu trả lời), trùng lặp, tokenize khứ hồi, độ dài.
Báo cáo `data_report.json` cho biết tỷ lệ dòng giữ được ở từng `MAX_LEN`; `MAX_LEN = None` chọn mức nhỏ nhất giữ được `KEEP_SHARE` số dòng
(dòng dài hơn bị loại, không bị cắt).

In [ ]:
check_dir = out.with_name(out.name + "_data_check")
try:
    tok_src = env.base_model() if env.can_train_bf16 else env.adapter("sql_rft_v3")   # thư mục adapter có cùng tokenizer + chat template
    env.run([py, "training/train_lora_bf16.py", "--train", train_file, *(["--dev", dev_file] if dev_file else []), "--model", tok_src,
             "--tokenizer", tok_src, "--out", check_dir, "--max-len", 65536, "--validate-only"],
            log=check_dir / "validate.log", show="train:|dev:|share of rows|sft_manifest|validate-only|Traceback|Error")
except RuntimeError as e:
    print("không kiểm tra được dữ liệu ở đây (cần transformers của training/requirements_bf16.txt và tokenizer):", e)
report = json.loads((check_dir / "data_report.json").read_text()) if (check_dir / "data_report.json").exists() else None
if report:
    t = report["train"]
    kept = {int(k): v for k, v in t["share_kept_at_max_len"].items()}
    max_len = MAX_LEN or next((L for L in sorted(kept) if L >= 16384 and kept[L] >= KEEP_SHARE), max(kept))
    display(pd.DataFrame({s: {"rows read": report[s]["rows_read"], "rows kept": report[s]["rows_kept"], "label share": f'{report[s]["label_share"]:.1%}',
                              "tokens p50 / p90 / max": "{p50} / {p90} / {max}".format(**report[s]["row_tokens"]),
                              "reply tokens p50 / max": "{p50} / {max}".format(**report[s]["reply_tokens"]), "rejected": report[s]["rejected"] or "-"}
                          for s in ("train", "dev") if report.get(s)}).T)
    ax = pd.Series(kept).plot.bar(figsize=(6, 2.5), title="share of train rows kept at each MAX_LEN", ylim=(0, 1.05))
    ax.axhline(KEEP_SHARE, ls="--", c="gray")
    print(f"MAX_LEN = {max_len}: giữ {kept.get(max_len, float('nan')):.1%} số dòng train")
else:
    max_len = MAX_LEN or 24576

### 2. Kernel fla

24 trên 32 lớp của Qwen3.5-9B là Gated DeltaNet. Có `fla-core` thì transformers dùng kernel Triton (nhanh hơn khoảng 90 lần mỗi lớp so với bản
PyTorch tham chiếu). Ô này so kernel với bản tham chiếu (forward và backward) trên dữ liệu ngẫu nhiên ở đúng kích thước của model. Trượt thì train
không có fla (đúng nhưng rất chậm).

In [ ]:
extra_env = {}
if not env.can_train_bf16:
    print("không có GPU bf16 >= 40 GB: bỏ qua kiểm tra kernel")
else:
    try:
        env.run([py, "training/check_fla.py", "--T", 4096], show="fla check|T=|rel L2|Traceback|Error")
    except RuntimeError as e:
        print("fla không dùng được ở đây, train không có fla:", e)
        extra_env = {"PYTHONPATH": str(env.root / "training/no_fla")}

In [ ]:
def train_cmd(out_dir, gpus=n_gpu, dev=dev_file, extra=(), report=REPORT_TO):
    launch = [py] if gpus == 1 else [py, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={gpus}"]
    cmd = launch + ["training/train_lora_bf16.py", "--train", train_file, *(["--dev", dev] if dev else []), "--model", model, "--out", out_dir,
                    "--max-len", max_len, "--r", R, "--alpha", ALPHA, "--dropout", DROPOUT, "--lr", lr, "--epochs", EPOCHS,
                    "--micro-batch", MICRO_BATCH, "--grad-accum", GRAD_ACCUM, "--warmup-ratio", WARMUP_RATIO, "--scheduler", SCHEDULER,
                    "--weight-decay", WEIGHT_DECAY, "--max-grad-norm", MAX_GRAD_NORM, "--eval-steps", EVAL_STEPS, "--save-steps", SAVE_STEPS,
                    "--early-stop", EARLY_STOP, "--dev-max-rows", DEV_MAX_ROWS, "--ce-chunk", CE_CHUNK, "--attn", ATTN, "--seed", SEED, "--report-to", report]
    cmd += (["--init-adapter", init] if init else []) + (["--limit-rows", LIMIT_ROWS] if LIMIT_ROWS else [])
    return cmd + (["--max-steps", MAX_STEPS] if MAX_STEPS > 0 and "--max-steps" not in extra else []) + list(extra)


model = env.base_model() if env.can_train_bf16 else None            # Qwen3.5-9B đúng revision đã nộp (~19 GB, tải một lần)
SHOW = "[train]|[eval]|train:|dev:|dev loss on|sft_manifest|Gated DeltaNet kernel|LoRA r=|rows per optimizer|resuming|continuing|steps in|adapter|Traceback|Error|OutOfMemory|out of memory"

### 3. Preflight bộ nhớ

Hai bước tối ưu, mỗi bước một dòng, trên **hai dòng dài nhất** (`--longest-first --grad-accum 1`), một GPU, không dev: đỉnh bộ nhớ thật ở `MAX_LEN` trước khi bỏ ra nhiều giờ.
Hết bộ nhớ thì giảm `MAX_LEN` (dòng dài hơn bị loại) hoặc dùng GPU lớn hơn; còn dư nhiều thì có thể tăng `MAX_LEN` (giữ nhiều dòng hơn).
Bỏ qua khi đang tiếp tục một lượt đã có checkpoint.

In [ ]:
resuming = RESUME and any(out.glob("checkpoint-*"))                 # tiếp tục một lượt đã chạy: bộ nhớ đã được kiểm chứng
if env.can_train_bf16 and PREFLIGHT and not resuming:
    pre = out.with_name(out.name + "_preflight")
    shutil.rmtree(pre, ignore_errors=True)
    try:
        env.run(train_cmd(pre, gpus=1, dev=None, extra=["--max-steps", 2, "--longest-first", "--grad-accum", 1, "--limit-rows", 0], report="none"),
                log=pre.with_suffix(".log"), env={**train_env, **extra_env}, show=SHOW, every=120)
        m = json.loads((pre / "run_manifest.json").read_text())
        use = m["peak_gpu_reserved_gib"] / m["gpu_total_gib"]
        print(f"preflight: {m['peak_gpu_gib']} GiB allocated, {m['peak_gpu_reserved_gib']} GiB reserved / {m['gpu_total_gib']} GiB ({use:.0%}), "
              f"{m['seconds'] / 2:.0f} s mỗi dòng dài nhất (bước đầu gồm cả autotune kernel)")
        if use > 0.95:
            print("=> sát giới hạn bộ nhớ: giảm MAX_LEN hoặc CE_CHUNK trước khi train thật")
    except RuntimeError as e:
        print(f"preflight thất bại ({e}). Xem {pre.with_suffix('.log')}: hết bộ nhớ thì giảm MAX_LEN (hiện {max_len}).")
        raise

### 4. Train

Nhật ký đầy đủ ở `OUT.log`; notebook chỉ hiện dòng loss / dev loss. Checkpoint được lưu mỗi `SAVE_STEPS` bước vào `OUT` (Kaggle:
`/kaggle/working/runs`, Colab: Google Drive, giữ lại sau phiên). Phiên bị ngắt: chạy lại notebook với `RESUME = True`, cùng dữ liệu và
thiết lập (`run_identity.json` kiểm tra điều này). Muốn bắt đầu lại từ đầu: đổi `OUT`.

In [ ]:
if env.can_train_bf16:
    env.run(train_cmd(out) + (["--resume"] if RESUME else []), log=out.with_name(out.name + ".log"), env={**train_env, **extra_env}, show=SHOW, every=300)
else:
    print("không có GPU bf16 >= 40 GB: bỏ qua bước train (A100, L40S, H100...). Các ô sau đọc kết quả nếu OUT đã có từ một máy khác.")

### 5. Đường loss

Train loss theo bước (mờ: từng bước, đậm: trung bình trượt), dev loss ở mỗi lần đánh giá, learning rate. Dev loss tăng trở lại trong khi train
loss vẫn giảm là dấu hiệu overfit: checkpoint tốt nhất (dev loss thấp nhất) là adapter cuối.

In [ ]:
state_file = out / "trainer_state.json"
if not state_file.exists():                                           # đang train hoặc bị ngắt: checkpoint mới nhất
    ckpts = sorted(out.glob("checkpoint-*/trainer_state.json"), key=lambda p: int(p.parent.name.split("-")[1]))
    state_file = ckpts[-1] if ckpts else None
if state_file:
    state = json.loads(state_file.read_text())
    hist = pd.DataFrame(state["log_history"])
    tr = hist.dropna(subset=["loss"]) if "loss" in hist else pd.DataFrame()
    ev = hist.dropna(subset=["eval_loss"]) if "eval_loss" in hist else pd.DataFrame()
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.5))
    if len(tr):
        a1.plot(tr.step, tr.loss, alpha=0.25, label="train loss")
        a1.plot(tr.step, tr.loss.rolling(max(1, len(tr) // 20), min_periods=1).mean(), label="train loss (smoothed)")
        a2.plot(tr.step, tr.learning_rate, c="gray")
    if len(ev):
        a1.plot(ev.step, ev.eval_loss, "o-", label="dev loss")
    a1.set(xlabel="step", title="loss"); a1.legend(); a2.set(xlabel="step", title="learning rate")
    plt.show()
    print(f"bước {state['global_step']} / {state['max_steps']}; dev loss tốt nhất {state.get('best_metric')} ({state.get('best_model_checkpoint')})")
else:
    print(f"chưa có trainer_state.json trong {out}")

### 6. Kiểm tra adapter

Trước khi đưa vào pipeline: r / alpha, đủ 248 module (496 tensor), tên tensor trùng với `adapters/sql_rft_v3` (vLLM nạp theo tên), và
`lora_B` khác 0 (adapter thật sự đã học). `run_manifest.json` ghi số bước, thời gian, loss, đỉnh bộ nhớ, phiên bản thư viện.

In [ ]:
import struct

import numpy as np


def st_tensors(path, names=None):
    """Header của .safetensors và (tùy chọn) vài tensor float32 đọc thẳng từ tệp: không cần torch trong kernel này."""
    with open(path, "rb") as f:
        n = struct.unpack("<Q", f.read(8))[0]
        head = json.loads(f.read(n))
        head.pop("__metadata__", None)
        arrays = {}
        for k in names or []:
            a, b = head[k]["data_offsets"]
            f.seek(8 + n + a)
            arrays[k] = np.frombuffer(f.read(b - a), dtype=np.float32)
    return head, arrays


adapter = out / "adapter"
if (adapter / "adapter_model.safetensors").exists():
    cfg = json.loads((adapter / "adapter_config.json").read_text())
    head, _ = st_tensors(adapter / "adapter_model.safetensors")
    b_keys = [k for k in head if "lora_B" in k and head[k]["dtype"] == "F32"]
    _, b = st_tensors(adapter / "adapter_model.safetensors", b_keys)
    base_cfg = (Path(model) if model else env.models / "Qwen3.5-9B") / "config.json"
    types = (json.loads(base_cfg.read_text())["text_config"]["layer_types"] if base_cfg.exists()
             else (["linear_attention"] * 3 + ["full_attention"]) * 8)                  # Qwen3.5-9B: 24 Gated DeltaNet + 8 full attention
    want = 2 * sum(7 if t == "full_attention" else 8 for t in types)   # lora_A + lora_B: 7 projections per attention layer, 8 per Gated DeltaNet layer
    ref = env.root / "adapters/sql_rft_v3/adapter_model.safetensors"
    same_names = set(head) == set(st_tensors(ref)[0]) if want == 496 and ref.exists() and ref.stat().st_size > 1_000_000 else None
    checks = {"r / alpha": f"{cfg['r']} / {cfg['lora_alpha']}", "tensors": f"{len(head)} / {want}", "names = sql_rft_v3": same_names,
              "lora_B khác 0": f"{sum(bool(np.abs(v).max() > 0) for v in b.values())} / {len(b)}",
              "MB": round((adapter / "adapter_model.safetensors").stat().st_size / 2**20, 1)}
    display(pd.Series(checks, name="adapter").to_frame())
    assert len(head) == want and same_names in (True, None), "adapter không khớp cách vLLM nạp sql_rft_v3"
    if (out / "run_manifest.json").exists():
        m = json.loads((out / "run_manifest.json").read_text())
        print({k: m.get(k) for k in ("steps", "seconds", "train_loss", "best_dev_loss", "best_checkpoint", "stopped_early", "peak_gpu_gib", "gpu", "versions")})
else:
    print(f"chưa có adapter trong {adapter}")

### 7. Dùng adapter vừa train

1. **Đo trên holdout (sạch, cần dữ liệu BTC trong `data/train`):** `02_inference` với `SPLIT = "train"`, `IDS = "splits/holdout600.txt"`, `REUSE_LAYERS = False`,
   `LORA = "runs/lora/qwen35_sft_a6000/adapter"` (`OUT` + `/adapter`). Chạy thêm một lượt với `LORA = "sql_rft_v3"` làm mốc.
2. **So sánh:** `03_evaluation` với `RUN` = lượt của adapter mới, `BASELINE` = lượt của v3: PA, chênh lệch theo shop, khoảng tin cậy bootstrap.
   v3 đã học 390 câu holdout nên điểm holdout của v3 bị thổi phồng; adapter mới thắng v3 trên holdout là bằng chứng mạnh.
3. **Fit cuối (tùy chọn):** khi đã chọn được số bước, train lại không giữ dev: với bộ SFT `data/data` đặt `DEV = None`; với dữ liệu của
   `04_training_data` đặt `DEV_SHOPS = 0` ở đó rồi `DEV = None` ở đây;
   `EPOCHS` giữ nguyên và `MAX_STEPS` = số bước của checkpoint tốt nhất, `OUT` mới.
4. **Nộp:** chạy `02_inference` trên `public_test` với `LORA = "runs/lora/qwen35_sft_a6000/adapter"`, so với bản trước bằng `03_evaluation` (phần so
   các bản nộp). Chỉ khi điểm public cao hơn bản đang nộp mới làm bước 8.

### 8. Đưa adapter BF16 vào bản nộp

Chỉ làm khi adapter mới có điểm public (chạy `FRESH=1`) cao hơn bản đang nộp. Chạy trên checkout local (trên Kaggle / Colab code là bản sao).
- `SUBMIT = False`: `scripts/use_adapter.py --dry-run` kiểm adapter (rank, alpha, `target_modules`, model gốc, tổng ≤ 15B) và liệt kê các chỗ sẽ
  sửa (`docker/Dockerfile`, `.dockerignore`, `docker/model_manifest.json`, `scripts/run_public.sh`, `scripts/run_holdout.sh`). Không ghi gì.
- `SUBMIT = True`: chép dữ liệu và cấu hình của lượt train vào `data/sft_bf16/` (train / dev, `split_manifest.json` hoặc `sft_manifest.json`, `train_args.json`, nhật ký)
  để `docker/train.sh` train lại đúng adapter này trong image (code train của bản nộp phải tái tạo được adapter), rồi đổi adapter nộp sang
  `adapters/NAME`. Sau đó: `python3 scripts/rule_check.py`, `python3 scripts/pack_training_code.py`, build Docker, commit (trọng số và
  `data/sft_bf16/*.jsonl` đi qua Git LFS).

In [ ]:
SUBMIT = False                 # True: chép dữ liệu vào data/sft_bf16/ và đổi adapter nộp. False: chỉ kiểm tra (dry-run)
NAME = "qwen35_sft_a6000"      # thư mục mới trong adapters/

adapter = out / "adapter"
if not (adapter / "adapter_model.safetensors").exists():
    print(f"chưa có adapter trong {adapter}")
else:
    if SUBMIT:
        dst = env.root / "data/sft_bf16"
        dst.mkdir(parents=True, exist_ok=True)
        files = {"train.jsonl": train_file, "dev.jsonl": dev_file, "split_manifest.json": train_file.parent / "split_manifest.json",
                 "sft_manifest.json": train_file.parent / "sft_manifest.json",
                 "train_args.json": out / "train_args.json", "run_manifest.json": out / "run_manifest.json", "trainer_state.json": out / "trainer_state.json",
                 "data_report.json": out / "data_report.json", "train.log": out.with_name(out.name + ".log")}
        for name, src in files.items():
            if src and Path(src).exists():
                shutil.copy2(src, dst / name)
        assert (dst / "train_args.json").exists(), "thiếu train_args.json của lượt train: docker/train.sh cần nó"
        print("data/sft_bf16:", sorted(p.name for p in dst.iterdir()))
    env.run([env.python, "scripts/use_adapter.py", adapter, "--name", NAME] + ([] if SUBMIT else ["--dry-run"]),
            show="->|reference|manifest|dry run|done|Error|error|adapters/")